In [2]:
# This notebook is adapted for local machine use (originally for Google Colab)
# Google Colab drive mounting is not needed - using local paths instead
import os
os.chdir('/Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection')
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection


check the dataset

In [3]:
import os

# Check if images and labels directories already exist
base_dir = os.getcwd()
images_dir = os.path.join(base_dir, 'images')
labels_dir = os.path.join(base_dir, 'labels')

if os.path.exists(images_dir) and os.path.exists(labels_dir):
    print(f"Dataset already exists in {base_dir}")
    print(f"Images: {os.listdir(images_dir)[:5]}...")  # Show first 5
    print(f"Labels: {os.listdir(labels_dir)[:5]}...")  # Show first 5
else:
    print("Warning: 'images' or 'labels' directory not found. Please ensure the dataset is in the current directory.")

Dataset already exists in /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection
Images: ['blue_bottle_blue_bottle_19659.jpg', 'phone_phone_1842297.jpg', 'red_cup_red_cup_3589425.jpg', 'phone_phone_410324.jpg', 'phone_a phone_388387.jpg']...
Labels: ['red_cup_red_cup_2603438.txt', 'blue_bottle_a blue water bottle_774466.txt', 'red_cup_red_cup_2786036.txt', 'phone_phone_1283938.txt', 'blue_bottle_blue_bottle_2408620.txt']...


Divide the training set and validation set

In [4]:
import os
import random
import shutil
from sklearn.model_selection import train_test_split

# Setup path - using current working directory
base_dir = os.getcwd()
images_dir = os.path.join(base_dir, 'images')
labels_dir = os.path.join(base_dir, 'labels')

# Create training and validation folders 创建训练和验证文件夹
train_images_dir = os.path.join(base_dir, 'train', 'images')
val_images_dir = os.path.join(base_dir, 'val', 'images')
train_labels_dir = os.path.join(base_dir, 'train', 'labels')
val_labels_dir = os.path.join(base_dir, 'val', 'labels')

for d in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
    os.makedirs(d, exist_ok=True)

# Get all image files 获取所有图片文件
image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
print(f"Found {len(image_files)} images to split")

# Divide the training set and validation set 划分训练集80%和验证集20%
train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

def copy_files(file_list, source_img_dir, source_label_dir, target_img_dir, target_label_dir):
    for f in file_list:
        # copy images
        src_img = os.path.join(source_img_dir, f)
        dst_img = os.path.join(target_img_dir, f)
        shutil.copy(src_img, dst_img)

        # Copy the corresponding label file (assuming the image and label file names are the same, only the suffixes are different) 复制对应的标签文件 (假设图片和标签文件名相同，只是后缀不同)
        label_file = os.path.splitext(f)[0] + '.txt'
        src_label = os.path.join(source_label_dir, label_file)
        dst_label = os.path.join(target_label_dir, label_file)
        if os.path.exists(src_label):
            shutil.copy(src_label, dst_label)
        else:
            print(f"Warning: Label file not found {src_label}")

# Perform copying 执行复制
copy_files(train_files, images_dir, labels_dir, train_images_dir, train_labels_dir)
copy_files(val_files, images_dir, labels_dir, val_images_dir, val_labels_dir)

print(f"Number of training set images: {len(os.listdir(train_images_dir))}")
print(f"Number of validation set images: {len(os.listdir(val_images_dir))}")

Found 135 images to split
Number of training set images: 130
Number of validation set images: 49


Create a dataset configuration file (data.yaml) 创建数据集配置文件 (data.yaml)

In [5]:
import yaml
import os

base_dir = os.getcwd()

data_yaml_content = f"""# Paths of training and validation images 训练和验证图片的路径
path: {base_dir}  # 数据集根目录
train: train/images  # 训练图片路径 (相对于 path)
val: val/images      # 验证图片路径 (相对于 path)

# Number of categories 类别数量
nc: 3

# Category names 类别名称
names: ['red_cup', 'blue_bottle', 'phone']
"""

yaml_path = os.path.join(base_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"data.yaml created successfully at {yaml_path}")

data.yaml created successfully at /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/data.yaml


 Train YOLO Model 训练 YOLO 模型


In [6]:
#1 Install Ultralytics YOLO (if not already installed)
import subprocess
import sys

try:
    from ultralytics import YOLO
    print("Ultralytics YOLO is already installed")
except ImportError:
    print("Installing ultralytics...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])

Ultralytics YOLO is already installed


In [7]:
from ultralytics import YOLO
import os
import numpy as np  
import torch
import time

# Verify PyTorch and numpy are available
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

base_dir = os.getcwd()
data_yaml_path = os.path.join(base_dir, 'data.yaml')

# Check if training data exists
if not os.path.exists(data_yaml_path):
    raise FileNotFoundError(f"data.yaml not found at {data_yaml_path}")


print("🎯 IMPROVED YOLO TRAINING - OPTION 2 WITH AUGMENTATION")
print("\n📊 Improvements Applied:")
print("  ✅ Larger Model: YOLOv8m instead of YOLOv8n (3x more parameters)")
print("  ✅ More Epochs: 50 instead of 10 (more training time)")
print("  ✅ Larger Input: 640 instead of 416 (better for small objects)")
print("  ✅ Aggressive Augmentation: rotation, flip, brightness, etc.")
print("  ✅ Better Optimization: SGD optimizer with tuned learning rate")
print("\n💡 Expected Results:")
print("  • mAP50: ~68% → 78-82% (+10-14%)")
print("  • Training Time: ~5-8 hours on MacBook CPU")
print("\n")

# Load the larger YOLOv8m model for better accuracy
# YOLOv8m has 3x more parameters than YOLOv8n, giving better detection
model = YOLO('yolov8m.pt')

# Start training with improved parameters optimized for MacBook CPU
start_time = time.time()
results = model.train(
    data=data_yaml_path,
    
    # 🎯 CORE IMPROVEMENTS
    epochs=50,                  # ⬆️ INCREASED: 10 → 50 (more training time for convergence)
    imgsz=640,                  # ⬆️ INCREASED: 416 → 640 (better for detecting small objects)
    batch=8,                    # OPTIMIZED FOR CPU: Keep at 8 for MacBook Pro
    patience=20,                # Early stopping if no improvement for 20 epochs
    
    # 💻 DEVICE SETTINGS
    device='cpu',               # MacBook CPU (no GPU)
    workers=2,                  # Data loading processes
    verbose=True,
    
    # 📈 DATA AUGMENTATION (The real game changer!)
    augment=True,               # Enable all augmentation
    mosaic=1.0,                 # Mix 4 images together (stronger augmentation)
    mixup=0.1,                  # Blend images (adds diversity)
    scale=0.5,                  # Scale variation ±50%
    fliplr=0.5,                 # Horizontal flip 50% of images
    flipud=0.5,                 # Vertical flip 50% of images
    degrees=15,                 # Rotate up to ±15 degrees
    translate=0.1,              # Translate up to 10% in x/y
    hsv_h=0.015,                # HSV hue variation
    hsv_s=0.7,                  # HSV saturation variation
    hsv_v=0.4,                  # HSV value (brightness) variation
    
    # 🔧 OPTIMIZATION SETTINGS
    optimizer='SGD',            # SGD often works better than Adam for detection
    lr0=0.01,                   # Initial learning rate (0.01 is standard)
    lrf=0.01,                   # Final learning rate ratio (10x smaller at end)
    momentum=0.937,             # Momentum for SGD
    weight_decay=0.0005,        # L2 regularization (prevents overfitting)
    
    # 📊 LOGGING & SAVING
    save=True,                  # Save best and last weights
    save_period=-1,             # Save only best model (don't save every epoch to save space)
    project='runs/detect',
    name='train',
)

elapsed_time = time.time() - start_time
hours = elapsed_time / 3600
print("✓ Training completed successfully!")
print(f"⏱️  Training took: {hours:.1f} hours ({elapsed_time/60:.0f} minutes)")

PyTorch version: 2.2.2
NumPy version: 1.26.4
🎯 IMPROVED YOLO TRAINING - OPTION 2 WITH AUGMENTATION

📊 Improvements Applied:
  ✅ Larger Model: YOLOv8m instead of YOLOv8n (3x more parameters)
  ✅ More Epochs: 50 instead of 10 (more training time)
  ✅ Larger Input: 640 instead of 416 (better for small objects)
  ✅ Aggressive Augmentation: rotation, flip, brightness, etc.
  ✅ Better Optimization: SGD optimizer with tuned learning rate

💡 Expected Results:
  • mAP50: ~68% → 78-82% (+10-14%)
  • Training Time: ~5-8 hours on MacBook CPU


Ultralytics 8.4.21 🚀 Python-3.9.23 torch-2.2.2 CPU (Intel Core i5-8259U 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/data.yaml, degrees=15, deterministic

PyTorch version: 2.2.2
NumPy version: 1.26.4
🎯 IMPROVED YOLO TRAINING - OPTION 2 WITH AUGMENTATION

📊 Improvements Applied:
  ✅ Larger Model: YOLOv8m instead of YOLOv8n (3x more parameters)
  ✅ More Epochs: 50 instead of 10 (more training time)
  ✅ Larger Input: 640 instead of 416 (better for small objects)
  ✅ Aggressive Augmentation: rotation, flip, brightness, etc.
  ✅ Better Optimization: SGD optimizer with tuned learning rate

💡 Expected Results:
  • mAP50: ~68% → 78-82% (+10-14%)
  • Training Time: ~5-8 hours on MacBook CPU


Ultralytics 8.4.21 🚀 Python-3.9.23 torch-2.2.2 CPU (Intel Core i5-8259U 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/data.yaml, degrees=15, deterministic

python(605) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(607) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(608) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


PyTorch version: 2.2.2
NumPy version: 1.26.4
🎯 IMPROVED YOLO TRAINING - OPTION 2 WITH AUGMENTATION

📊 Improvements Applied:
  ✅ Larger Model: YOLOv8m instead of YOLOv8n (3x more parameters)
  ✅ More Epochs: 50 instead of 10 (more training time)
  ✅ Larger Input: 640 instead of 416 (better for small objects)
  ✅ Aggressive Augmentation: rotation, flip, brightness, etc.
  ✅ Better Optimization: SGD optimizer with tuned learning rate

💡 Expected Results:
  • mAP50: ~68% → 78-82% (+10-14%)
  • Training Time: ~5-8 hours on MacBook CPU


Ultralytics 8.4.21 🚀 Python-3.9.23 torch-2.2.2 CPU (Intel Core i5-8259U 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/data.yaml, degrees=15, deterministic

python(605) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(607) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(608) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


...

PyTorch version: 2.2.2
NumPy version: 1.26.4
🎯 IMPROVED YOLO TRAINING - OPTION 2 WITH AUGMENTATION

📊 Improvements Applied:
  ✅ Larger Model: YOLOv8m instead of YOLOv8n (3x more parameters)
  ✅ More Epochs: 50 instead of 10 (more training time)
  ✅ Larger Input: 640 instead of 416 (better for small objects)
  ✅ Aggressive Augmentation: rotation, flip, brightness, etc.
  ✅ Better Optimization: SGD optimizer with tuned learning rate

💡 Expected Results:
  • mAP50: ~68% → 78-82% (+10-14%)
  • Training Time: ~5-8 hours on MacBook CPU


Ultralytics 8.4.21 🚀 Python-3.9.23 torch-2.2.2 CPU (Intel Core i5-8259U 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/data.yaml, degrees=15, deterministic

python(605) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(607) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(608) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Overriding model.yaml nc=80 with nc=3

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

python(628) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 135.1±32.9 MB/s, size: 54.8 KB)
val: Scanning /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/labels... 49 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 49/49 630.4it/s 0.1s
val: New cache created: /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/labels.cache
optimizer: SGD(lr=0.01, momentum=0.937) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Plotting labels to /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/runs/detect/runs/detect/train/labels.jpg... 
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/runs/detect/runs/detect/train
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50         0G      2.163      3.809      2.449         24        640: 100% ━━━━━━━━━━━━ 17/17 43.5s

Evaluate and Test 评估和测试

In [15]:
# Load the trained best model 加载训练好的最佳模型
from ultralytics import YOLO
import os

base_dir = os.getcwd()
best_model_path = os.path.join(base_dir, 'runs', 'detect', 'runs', 'detect', 'train', 'weights', 'best.pt')

if os.path.exists(best_model_path):
    best_model = YOLO(best_model_path)
    data_yaml_path = os.path.join(base_dir, 'data.yaml')
    
    # Evaluate on the validation set 在验证集上评估
    metrics = best_model.val(data=data_yaml_path)
    print(metrics)
else:
    print(f"Model not found at {best_model_path}. Please train the model first.")

Ultralytics 8.4.21 🚀 Python-3.9.23 torch-2.2.2 CPU (Intel Core i5-8259U 2.30GHz)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 131.8±25.8 MB/s, size: 54.7 KB)
val: Scanning /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/labels.cache... 49 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 49/49 71.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 18.5s/it 1:1436.4s
                   all         49         65      0.906      0.954      0.963      0.632
               red_cup         15         22       0.89          1      0.975      0.618
           blue_bottle         22         22       0.95      0.909      0.936      0.593
                 phone         12         21      0.879      0.952      0.976      0.685
Speed: 6.0ms preprocess, 1473.8ms inference, 0.0ms loss, 5.8ms postprocess per image
Results sa

In [16]:

# Inference with Adjustable Confidence Threshold (OPTION 4)
# Lower confidence = catch more objects (higher recall)
# Higher confidence = fewer but more reliable detections (higher precision)

from ultralytics import YOLO
import cv2
import os
import matplotlib.pyplot as plt
from pathlib import Path

base_dir = os.getcwd()
best_model_path = os.path.join(base_dir, 'runs', 'detect', 'runs', 'detect', 'train', 'weights', 'best.pt')

if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    
    # Test on a validation image
    val_images_dir = os.path.join(base_dir, 'val', 'images')
    if os.path.exists(val_images_dir):
        image_files = [f for f in os.listdir(val_images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
        
        if image_files:
            print("🎯 TESTING DIFFERENT CONFIDENCE THRESHOLDS")
            print("\n📊 How Confidence Threshold Affects Detection:")
            print("  • Low (0.1):  Catch MORE objects (more false positives)")
            print("  • Medium (0.25): Balanced (default)")
            print("  • High (0.5):  Only confident detections (fewer misses)\n")
            
            test_image_path = os.path.join(val_images_dir, image_files[0])
            print(f"Testing on: {image_files[0]}\n")
            
            # Test different confidence thresholds
            confidence_levels = [0.1, 0.25, 0.5]
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            fig.suptitle('Detection Results at Different Confidence Thresholds', fontsize=14, fontweight='bold')
            
            for idx, conf in enumerate(confidence_levels):
                results = model(test_image_path, conf=conf)
                annotated = results[0].plot()
                num_detections = len(results[0].boxes)
                
                # Calculate detection details
                class_counts = {}
                avg_confidence = 0
                for box in results[0].boxes:
                    class_id = int(box.cls[0])
                    class_name = model.names[class_id]
                    class_counts[class_name] = class_counts.get(class_name, 0) + 1
                    avg_confidence += float(box.conf[0])
                
                if num_detections > 0:
                    avg_confidence /= num_detections
                
                # Display
                axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
                title = f"\nConfidence: {conf:.0%}\nDetections: {num_detections}"
                if num_detections > 0:
                    title += f"\nAvg Confidence: {avg_confidence:.0%}"
                axes[idx].set_title(title, fontweight='bold')
                axes[idx].axis('off')
                
                # Print details
                print(f"Confidence Threshold: {conf:.0%}")
                print(f"  ├─ Total detections: {num_detections}")
                if num_detections > 0:
                    print(f"  ├─ Average confidence: {avg_confidence:.2%}")
                    for class_name, count in sorted(class_counts.items()):
                        print(f"  ├─ {class_name}: {count}")
                print()
            
            plt.tight_layout()
            plt.show()
            
            print("="*70)
            print("💡 WHICH THRESHOLD TO USE:")
            print("="*70)
            print("""
🎯 FOR MY USE CASE (since recall is 70%, missing some objects):

Recommendation: Use confidence = 0.25 (Default)
    ✓ Balanced precision and recall
    ✓ Good for catching most objects
    
OR for higher recall (catch everything):
    → Lower to 0.15-0.2 (more detections, some false positives)
    
OR for higher precision (only sure detections):
    → Raise to 0.3-0.5 (fewer false positives, might miss some)
            """)
        else:
            print("⚠️  No validation images found")
    else:
        print(f"⚠️  Validation directory not found: {val_images_dir}")
else:
    print(f"⚠️  Model not found at: {best_model_path}")
    print("   Train the model first (Cell 10) before testing")

🎯 TESTING DIFFERENT CONFIDENCE THRESHOLDS

📊 How Confidence Threshold Affects Detection:
  • Low (0.1):  Catch MORE objects (more false positives)
  • Medium (0.25): Balanced (default)
  • High (0.5):  Only confident detections (fewer misses)

Testing on: phone_phone_410324.jpg


image 1/1 /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/images/phone_phone_410324.jpg: 448x640 1 blue_bottle, 3310.3ms
Speed: 145.5ms preprocess, 3310.3ms inference, 58.0ms postprocess per image at shape (1, 3, 448, 640)
Confidence Threshold: 10%
  ├─ Total detections: 1
  ├─ Average confidence: 89.28%
  ├─ blue_bottle: 1


image 1/1 /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/images/phone_phone_410324.jpg: 448x640 1 blue_bottle, 1000.9ms
Speed: 8.7ms preprocess, 1000.9ms inference, 5.2ms postprocess per image at shape (1, 3, 448, 640)
Confidence Threshold: 25%
  ├─ Total detections: 1
  ├─ Average confidence: 89.28%
  ├─ blue_bottle: 1


image 1/1 /Users/ron/Desktop

🎯 TESTING DIFFERENT CONFIDENCE THRESHOLDS

📊 How Confidence Threshold Affects Detection:
  • Low (0.1):  Catch MORE objects (more false positives)
  • Medium (0.25): Balanced (default)
  • High (0.5):  Only confident detections (fewer misses)

Testing on: phone_phone_410324.jpg


image 1/1 /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/images/phone_phone_410324.jpg: 448x640 1 blue_bottle, 3310.3ms
Speed: 145.5ms preprocess, 3310.3ms inference, 58.0ms postprocess per image at shape (1, 3, 448, 640)
Confidence Threshold: 10%
  ├─ Total detections: 1
  ├─ Average confidence: 89.28%
  ├─ blue_bottle: 1


image 1/1 /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/images/phone_phone_410324.jpg: 448x640 1 blue_bottle, 1000.9ms
Speed: 8.7ms preprocess, 1000.9ms inference, 5.2ms postprocess per image at shape (1, 3, 448, 640)
Confidence Threshold: 25%
  ├─ Total detections: 1
  ├─ Average confidence: 89.28%
  ├─ blue_bottle: 1


image 1/1 /Users/ron/Desktop

<Figure size 1800x500 with 3 Axes>

💡 WHICH THRESHOLD TO USE:

🎯 FOR MY USE CASE (since recall is 70%, missing some objects):

Recommendation: Use confidence = 0.25 (Default)
    ✓ Balanced precision and recall
    ✓ Good for catching most objects
    
OR for higher recall (catch everything):
    → Lower to 0.15-0.2 (more detections, some false positives)
    
OR for higher precision (only sure detections):
    → Raise to 0.3-0.5 (fewer false positives, might miss some)
            


# ✨ IMPROVED REAL-TIME DETECTION (Options 1 & 2 Combined)

**Lower Confidence Threshold for Better Recall (Catch More Objects)**
- Default: conf=0.25
- Better Recall: conf=0.15 or 0.2
- Test the thresholds in the cell above ⬆️

In [18]:
import cv2
from ultralytics import YOLO
import os
import time
import numpy as np
from collections import deque

base_dir = os.getcwd()
best_model_path = os.path.join(base_dir, 'runs', 'detect', 'runs', 'detect', 'train', 'weights', 'best.pt')

# ============================================================================
# OPTION 1: IMPROVED WEBCAM DETECTION (with adjustable confidence)
# ============================================================================

def run_improved_webcam_detection(confidence_threshold=0.25, max_frames=None):
    """
    Real-time detection with better recall (catches more objects).
    
    Args:
        confidence_threshold: Detection confidence (0.1=high recall, 0.5=high precision)
        max_frames: Stop after N frames (None=unlimited, press 'q' to quit)
    """
    
    if not os.path.exists(best_model_path):
        print(f"❌ Model not found. Train first: {best_model_path}")
        return
    
    model = YOLO(best_model_path)
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("❌ Cannot open webcam")
        return
    
    frame_times = deque(maxlen=30)
    frame_count = 0
    detection_stats = {}
    
    print("\n" + "="*70)
    print("🎬 IMPROVED WEBCAM DETECTION - OPTION 1")
    print("="*70)
    print(f"Confidence Threshold: {confidence_threshold:.0%}")
    print(f"Benefits: Catch more objects (higher recall)")
    print(f"Press 'q' to stop\n")
    
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            # Run detection with custom confidence
            start_time = time.time()
            results = model(frame, conf=confidence_threshold, verbose=False)
            inference_time = time.time() - start_time
            frame_times.append(inference_time)
            
            # Process detections
            annotated_frame = results[0].plot()
            for box in results[0].boxes:
                class_id = int(box.cls[0])
                class_name = model.names[class_id]
                detection_stats[class_name] = detection_stats.get(class_name, 0) + 1
            
            # Calculate metrics
            detections = len(results[0].boxes)
            avg_time = np.mean(frame_times)
            fps = 1 / avg_time if avg_time > 0 else 0
            
            # Add info to frame
            info_text = f"FPS: {fps:.1f} | Objects: {detections} | Conf: {confidence_threshold:.0%}"
            cv2.putText(annotated_frame, info_text, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            # Show frame
            cv2.imshow('Improved YOLO Detection (Press q to stop)', annotated_frame)
            
            frame_count += 1
            if frame_count % 30 == 0:
                print(f"Processed {frame_count} frames | Avg FPS: {fps:.1f}")
            
            if max_frames and frame_count >= max_frames:
                break
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    
    finally:
        cap.release()
        cv2.destroyAllWindows()
    
    # Print summary
    print("\n" + "="*70)
    print("📊 DETECTION SUMMARY")
    print("="*70)
    print(f"Total frames: {frame_count}")
    print(f"Total objects detected: {sum(detection_stats.values())}")
    print(f"Average FPS: {1/np.mean(frame_times) if frame_times else 0:.1f}")
    print(f"\nDetections by class:")
    for class_name, count in sorted(detection_stats.items()):
        pct = 100 * count / max(sum(detection_stats.values()), 1)
        print(f"  {class_name:20}: {count:3d} ({pct:5.1f}%)")
    print("="*70)


# ============================================================================
# OPTION 2: IMPROVED VIDEO FILE DETECTION (with adjustable confidence)
# ============================================================================

def run_improved_video_detection(video_path, confidence_threshold=0.25, save_output=False):
    """
    Process video file with custom confidence threshold.
    
    Args:
        video_path: Path to video file
        confidence_threshold: Detection confidence
        save_output: Save annotated video
    """
    
    if not os.path.exists(video_path):
        print(f"❌ Video not found: {video_path}")
        return
    
    if not os.path.exists(best_model_path):
        print(f"❌ Model not found: {best_model_path}")
        return
    
    model = YOLO(best_model_path)
    cap = cv2.VideoCapture(video_path)
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    video_writer = None
    if save_output:
        output_path = os.path.join(base_dir, f"detected_{os.path.basename(video_path)}")
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    frame_times = deque(maxlen=30)
    frame_count = 0
    detection_stats = {}
    
    print("\n" + "="*70)
    print("🎬 IMPROVED VIDEO DETECTION - OPTION 2")
    print("="*70)
    print(f"Video: {os.path.basename(video_path)}")
    print(f"Confidence Threshold: {confidence_threshold:.0%}")
    print(f"Size: {width}x{height} @ {fps}fps\n")
    
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            start_time = time.time()
            results = model(frame, conf=confidence_threshold, verbose=False)
            inference_time = time.time() - start_time
            frame_times.append(inference_time)
            
            annotated_frame = results[0].plot()
            
            for box in results[0].boxes:
                class_id = int(box.cls[0])
                class_name = model.names[class_id]
                detection_stats[class_name] = detection_stats.get(class_name, 0) + 1
            
            detections = len(results[0].boxes)
            avg_time = np.mean(frame_times)
            fps_current = 1 / avg_time if avg_time > 0 else 0
            
            info_text = f"FPS: {fps_current:.1f} | Objects: {detections}"
            cv2.putText(annotated_frame, info_text, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            if video_writer:
                video_writer.write(annotated_frame)
            
            frame_count += 1
            if frame_count % 30 == 0:
                print(f"Processed {frame_count} frames ({frame_count//fps}s) | Avg FPS: {fps_current:.1f}")
    
    finally:
        cap.release()
        if video_writer:
            video_writer.release()
    
    # Print summary
    print("\n" + "="*70)
    print("📊 VIDEO DETECTION SUMMARY")
    print("="*70)
    print(f"Total frames processed: {frame_count}")
    print(f"Total objects detected: {sum(detection_stats.values())}")
    print(f"Average FPS: {1/np.mean(frame_times) if frame_times else 0:.1f}")
    print(f"\nDetections by class:")
    for class_name, count in sorted(detection_stats.items()):
        pct = 100 * count / max(sum(detection_stats.values()), 1)
        print(f"  {class_name:20}: {count:3d} ({pct:5.1f}%)")
    
    if save_output:
        print(f"\n✓ Output saved: {os.path.join(base_dir, f'detected_{os.path.basename(video_path)}')}")
    print("="*70)


# ============================================================================
# USAGE EXAMPLES (UNCOMMENT TO RUN)
# ============================================================================

print("📝 USAGE EXAMPLES:\n")
print(run_improved_webcam_detection(confidence_threshold=0.25))
# Option 1: Test on webcam (10 seconds = ~300 frames)
# run_improved_webcam_detection(confidence_threshold=0.2, max_frames=300)

# Option 2: Test on webcam with default confidence
# run_improved_webcam_detection(confidence_threshold=0.25)

# Option 2: Process a video file
# run_improved_video_detection('path/to/video.mp4', confidence_threshold=0.2, save_output=True)

#💡 TIP: Lower confidence (0.15-0.2) catches more objects but has more false positives.
#       Higher confidence (0.4-0.5) is more precise but might miss some objects.


📝 USAGE EXAMPLES:



📝 USAGE EXAMPLES:



2026-03-12 23:16:55.706 python[93459:1680198] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


📝 USAGE EXAMPLES:



2026-03-12 23:16:55.706 python[93459:1680198] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.



🎬 IMPROVED WEBCAM DETECTION - OPTION 1
Confidence Threshold: 25%
Benefits: Catch more objects (higher recall)
Press 'q' to stop

Processed 30 frames | Avg FPS: 1.8
Processed 60 frames | Avg FPS: 2.3
Processed 90 frames | Avg FPS: 2.7
Processed 120 frames | Avg FPS: 2.6
Processed 150 frames | Avg FPS: 2.4

📊 DETECTION SUMMARY
Total frames: 152
Total objects detected: 171
Average FPS: 1.8

Detections by class:
  blue_bottle         : 130 ( 76.0%)
  red_cup             :  41 ( 24.0%)
None


# 📊 Summary of Accuracy Improvements Applied

## ✨ Quick Wins Applied (Easy, 30 min - 2 hours)

| Improvement | Before | After | Impact |
|---|---|---|---|
| **Model Size** | YOLOv8n (nano) | YOLOv8m (medium) | 3x more parameters → Better accuracy |
| **Training Epochs** | 10 | 50 | More training time for convergence |
| **Input Size** | 416×416 | 640×640 | Better detection of small objects |
| **Batch Size** | 8 | 8 | Optimized for MacBook CPU |
| **Confidence Threshold** | 0.25 (fixed) | Adjustable | Can tune for your needs |

**Expected Accuracy Improvement:** mAP50: ~68% → **78-82%** (+10-14%)

---

## 🎯 Medium Effort Applied (2 - 8 hours)

### Data Augmentation (The Real Game Changer!)
- **Mosaic:** Mixing 4 images together → Creates diverse training examples
- **Mixup:** Blending images → Adds variety
- **Rotation:** ±15 degrees → Handle angled objects
- **Flip:** Horizontal & vertical → Learn reflections
- **Brightness:** HSV changes → Handle lighting variations
- **Scale & Translate:** Size & position variations → Robustness

### Optimizer Improvements
- **Optimizer:** Changed from default Adam → SGD (better for detection)
- **Learning Rate:** 0.01 initial, 0.01 final ratio → Controlled learning
- **Momentum:** 0.937 → Faster convergence
- **Weight Decay:** 0.0005 → Prevents overfitting

---

## 🚀 Expected Training Time

**Your MacBook Pro (CPU only):**
- Original: ~2 hours for 10 epochs = 12 minutes/epoch
- New version: ~5-8 hours for 50 epochs = 6-10 minutes/epoch (better optimization)
- Can run overnight while you sleep!

---

## 📈 How to Use the Improved Code

### Step 1: Train with Improved Parameters
Run **Cell 10** (Training cell with improvements):
```python
results = model.train(...)  # Uses YOLOv8m, 50 epochs, augmentation
```

### Step 2: Compare Thresholds
Run **Cell 12** (Confidence threshold comparison):
```python
# Shows detection difference at conf=0.1, 0.25, 0.5
# Helps you pick best threshold
```

### Step 3: Test Real-Time Detection
Run **Cell 14** (Improved detection):
```python
# Option 1: Webcam with adjustable confidence
run_improved_webcam_detection(confidence_threshold=0.2, max_frames=300)

# Option 2: Video file
run_improved_video_detection('video.mp4', confidence_threshold=0.2, save_output=True)
```

---

## 💡 Key Tips for MacBook Pro CPU

✅ **Keep batch=8** - Balance between speed and stability  
✅ **workers=2** - Data loading efficiency  
✅ **Use patience=20** - Early stopping if model stops improving  
✅ **augment=True** - Essential for small datasets (150 images)  

⚠️ **Don't:**
- Use batch > 16 (will be very slow)
- Use imgsz > 768 (diminishing returns)
- Skip augmentation (crucial for 150 images!)

---

## 🎓 Understanding Confidence Threshold

When recall is **70%** (missing some objects), **lower the confidence:**

| Threshold | Precision | Recall | Use When |
|---|---|---|---|
| 0.1 | Lower | Higher | Want to catch ALL objects |
| 0.2 | Good | Good | **Recommended for your case** |
| 0.25 | Good | Good | Default |
| 0.4+ | Higher | Lower | Want only super confident detections |

**For your dataset:** Try 0.15-0.2 to improve recall (catch more blue bottles, phones, red cups)

---

## 📊 What Changed in Your Code

### Cell 10 (Training):
- ✅ Model: `YOLO('yolov8n.pt')` → `YOLO('yolov8m.pt')`
- ✅ Epochs: `epochs=10` → `epochs=50`
- ✅ Input Size: `imgsz=416` → `imgsz=640`
- ✅ Added augmentation parameters (mosaic, mixup, rotation, etc.)
- ✅ Changed optimizer to SGD with better hyperparameters
- ✅ Added time tracking to see actual training duration

### New Cells:
- **Cell 12:** Confidence threshold comparison tool
- **Cell 14:** Improved detection functions (webcam + video with adjustable confidence)

---

## ✅ Next Steps

1. **Run Cell 10** to train with improvements (~5-8 hours overnight)
2. **Run Cell 12** to compare confidence thresholds
3. **Run Cell 14** to test on webcam/video with best threshold
4. **Compare results** with your original training
5. **Iterate:** If still not good enough, collect more data or adjust further

**Expected:** mAP50 improvement from ~68% → 78-82% 🎯

Detection image from validation set

In [ ]:
import cv2
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
from IPython.display import clear_output

base_dir = os.getcwd()
best_model_path = os.path.join(base_dir, base_dir, 'runs', 'detect', 'runs', 'detect', 'train', 'weights', 'best.pt')

# Load your trained model
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    
    # Example: Run detection on a sample image from validation set
    val_images_dir = os.path.join(base_dir, 'val', 'images')
    if os.path.exists(val_images_dir):
        image_files = [f for f in os.listdir(val_images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
        
        if image_files:
            # Test on first image
            test_image_path = os.path.join(val_images_dir, image_files[0])
            print(f"Testing on image: {test_image_path}")
            
            results = model(test_image_path)
            annotated = results[0].plot()
            
            # Display result using matplotlib
            plt.figure(figsize=(12, 8))
            plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
            plt.title("YOLO Detection Result")
            plt.axis('off')
            plt.tight_layout()
            plt.show()
        else:
            print("No images found in validation directory")
    else:
        print(f"Validation images directory not found at {val_images_dir}")
else:
    print(f"Model not found at {best_model_path}. Please train the model first.")
    

Testing on image: /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/images/phone_phone_410324.jpg

image 1/1 /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/images/phone_phone_410324.jpg: 448x640 1 blue_bottle, 1015.8ms
Speed: 9.3ms preprocess, 1015.8ms inference, 3.6ms postprocess per image at shape (1, 3, 448, 640)


Testing on image: /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/images/phone_phone_410324.jpg

image 1/1 /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/images/phone_phone_410324.jpg: 448x640 1 blue_bottle, 1015.8ms
Speed: 9.3ms preprocess, 1015.8ms inference, 3.6ms postprocess per image at shape (1, 3, 448, 640)


<Figure size 1200x800 with 1 Axes>

Testing on image: /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/images/phone_phone_410324.jpg

image 1/1 /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/val/images/phone_phone_410324.jpg: 448x640 1 blue_bottle, 1015.8ms
Speed: 9.3ms preprocess, 1015.8ms inference, 3.6ms postprocess per image at shape (1, 3, 448, 640)


<Figure size 1200x800 with 1 Axes>

: 

 Real-time Demo 实时演示

In [12]:
import cv2
from ultralytics import YOLO
import os
import time
import numpy as np
from collections import deque

base_dir = os.getcwd()
best_model_path = os.path.join(base_dir, 'runs', 'detect', 'train4', 'weights', 'best.pt')

def run_detector_on_video(model, video_source=0, output_path=None, confidence_threshold=0.5, max_frames=None):
    """
    Run YOLO detector on webcam or video file with real-time predictions.
    
    Args:
        model: YOLO model object
        video_source: 0 for webcam, or path to video file (str)
        output_path: Path to save output video (optional)
        confidence_threshold: Minimum confidence for detections (0.0 to 1.0)
        max_frames: Maximum frames to process (None for all)
    """
    cap = cv2.VideoCapture(video_source)
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS)) if isinstance(video_source, str) else 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Initialize video writer if output path is provided
    video_writer = None
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Performance metrics
    frame_times = deque(maxlen=30)  # Store last 30 frame times
    frame_count = 0
    detection_count = 0
    
    source_type = "Webcam" if video_source == 0 else f"Video: {os.path.basename(video_source)}"
    print(f"Starting detection on {source_type}")
    print(f"Resolution: {width}x{height} @ {fps}fps")
    print("Press 'q' to stop\n")
    
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            # Run detection (YOLO's single-pass architecture)
            start_time = time.time()
            results = model(frame, conf=confidence_threshold, verbose=False)
            inference_time = time.time() - start_time
            frame_times.append(inference_time)
            
            # Visualize results
            annotated_frame = results[0].plot()
            
            # Calculate metrics
            detections = len(results[0].boxes)
            detection_count += detections
            
            # Calculate FPS
            avg_time = np.mean(frame_times)
            fps_display = 1 / avg_time if avg_time > 0 else 0
            
            # Add performance info to frame
            info_text = f"FPS: {fps_display:.1f} | Detections: {detections} | Time: {inference_time*1000:.1f}ms"
            cv2.putText(annotated_frame, info_text, (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            # Display frame
            cv2.imshow('YOLO Real-Time Detection', annotated_frame)
            
            # Write to output video if specified
            if video_writer:
                video_writer.write(annotated_frame)
            
            frame_count += 1
            
            # Print progress
            if frame_count % 30 == 0:
                print(f"Processed {frame_count} frames | Avg inference time: {avg_time*1000:.1f}ms | Avg FPS: {fps_display:.1f}")
            
            # Check for max frames
            if max_frames and frame_count >= max_frames:
                break
            
            # Press 'q' to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
                
    finally:
        cap.release()
        if video_writer:
            video_writer.release()
        cv2.destroyAllWindows()
    
    # Print summary
    print(f"Detection Complete Summary:")
    print(f"Total frames processed: {frame_count}")
    print(f"Total detections: {detection_count}")
    print(f"Avg detections per frame: {detection_count/frame_count:.2f}")
    print(f"Avg inference time per frame: {np.mean(frame_times)*1000:.1f}ms")
    print(f"Average FPS: {1/np.mean(frame_times):.1f}")
    print(f"\nYOLO's Architecture Advantage:")
    print(f"- Single-pass detection (original YOLO innovation, ~{inference_time*1000:.1f}ms per frame)")
    print(f"- Introduced by Joseph Redmon et al. for real-time object detection")
    print(f"- Processes entire image in one neural network evaluation")
    if output_path:
        print(f"\nOutput video saved to: {output_path}")


# Check if model exists
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    
    # Example 1: Run on webcam (uncomment to use)
    # Uncomment the line below to run on my webcam (COMMENTED OUT - uncomment to enable)
    # To quit: Press 'q' in the window
    # run_detector_on_video(model, video_source=0, confidence_threshold=0.5)
    
    # Example 2: Run on a video file 
    # Uncomment and modify the path below to run on a video file
    # video_path = os.path.join(base_dir, 'test_video.mp4')
    # output_path = os.path.join(base_dir, 'detection_output.mp4')
    # run_detector_on_video(model, video_source=video_path, output_path=output_path, confidence_threshold=0.5)
    
    print("Real-time detection function is ready!")
    print("\nTo run detection on my webcam, uncomment and execute:")
    print("  run_detector_on_video(model, video_source=0, confidence_threshold=0.5)")
    print("\nTo run detection on a video file, uncomment and execute:")
    print("  run_detector_on_video(model, video_source='path/to/video.mp4', output_path='output.mp4')")
    
else:
    print(f"Model not found at {best_model_path}")
    print("Please train the model first.")

Real-time detection function is ready!

To run detection on my webcam, uncomment and execute:
  run_detector_on_video(model, video_source=0, confidence_threshold=0.5)

To run detection on a video file, uncomment and execute:
  run_detector_on_video(model, video_source='path/to/video.mp4', output_path='output.mp4')


In [13]:
# Testing optional

# ============================================================================
# OPTION 1: Run on Webcam (Real-time detection from your camera)
# ============================================================================
# Uncomment the line below to run detection on your webcam
# Press 'q' to stop the detection

# run_detector_on_video(model, video_source=0, confidence_threshold=0.5)


# ============================================================================
# OPTION 2: Run on a Video File
# ============================================================================
# First, specify my video file path, then uncomment to run

# video_file = '/path/to/my/video.mp4'  # Change this to my video path
# output_video = os.path.join(base_dir, 'detection_output.mp4')
# run_detector_on_video(model, video_source=video_file, output_path=output_video, confidence_threshold=0.5)


# ============================================================================
# OPTION 3: Run on a Single Image ( test one image)
# ============================================================================


def detect_on_image(model, image_path, confidence_threshold=0.5):
    """Run detection on a single image and display result."""
    import matplotlib.pyplot as plt
    
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not load image from {image_path}")
        return
    
    print(f"Running detection on: {image_path}")
    
    # Run detection
    results = model(image, conf=confidence_threshold)
    annotated = results[0].plot()
    
    # Display
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"YOLO Detection - {len(results[0].boxes)} objects detected")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Print detection details
    print(f"\nDetections found: {len(results[0].boxes)}")
    for i, box in enumerate(results[0].boxes):
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        class_name = model.names[class_id]
        print(f"  {i+1}. {class_name}: {confidence:.2%} confidence")

# To use OPTION 3, uncomment and modify the line below:
image_path = os.path.join(base_dir, 'test_image.jpg')
detect_on_image(model, image_path, confidence_threshold=0.5)  # Uncomment this line to run


print("QUICKSTART GUIDE")
print("\n📹 OPTION 1 - Webcam Detection (Real-time):")
print("   Uncomment and run: run_detector_on_video(model, video_source=0)")
print("\n📽️  OPTION 2 - Video File Detection:")
print("   1. Set your video path: video_file = '/path/to/video.mp4'")
print("   2. Uncomment and run: run_detector_on_video(model, video_source=video_file)")
print("\n🖼️  OPTION 3 - Single Image Detection:")
print("   1. Set test image path: image_path = '/path/to/image.jpg'")
print("   2. Uncomment and run: detect_on_image(model, image_path)")
print("\n💡 KEY FEATURES:")
print("   ✓ Real-time FPS calculation")
print("   ✓ Per-frame inference time display")
print("   ✓ Detection count and confidence scores")
print("   ✓ Video output saving (Option 2)")
print("   ✓ YOLO's single-pass detection architecture")

Running detection on: /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/test_image.jpg

0: 320x416 1 phone, 138.3ms
Speed: 2.8ms preprocess, 138.3ms inference, 2.1ms postprocess per image at shape (1, 3, 320, 416)


Running detection on: /Users/ron/Desktop/Deeplearning/3.10_lab_yolo_object_detection/test_image.jpg

0: 320x416 1 phone, 138.3ms
Speed: 2.8ms preprocess, 138.3ms inference, 2.1ms postprocess per image at shape (1, 3, 320, 416)


<Figure size 1400x800 with 1 Axes>


Detections found: 1
  1. phone: 78.21% confidence
QUICKSTART GUIDE

📹 OPTION 1 - Webcam Detection (Real-time):
   Uncomment and run: run_detector_on_video(model, video_source=0)

📽️  OPTION 2 - Video File Detection:
   1. Set your video path: video_file = '/path/to/video.mp4'
   2. Uncomment and run: run_detector_on_video(model, video_source=video_file)

🖼️  OPTION 3 - Single Image Detection:
   1. Set test image path: image_path = '/path/to/image.jpg'
   2. Uncomment and run: detect_on_image(model, image_path)

💡 KEY FEATURES:
   ✓ Real-time FPS calculation
   ✓ Per-frame inference time display
   ✓ Detection count and confidence scores
   ✓ Video output saving (Option 2)
   ✓ YOLO's single-pass detection architecture
